You’re optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk. Each recipe is evaluated with a combined score based on flavour, consistency, calories, waste and cost, where each factor contributes negative points as judged by an expert taster. This means the total score is negative by design. To frame this as a maximisation problem, your goal is to bring that score as close to zero as possible or, equivalently, to maximise the negative of the total sum.

In [1]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================
# Core numerics, scikit-learn Gaussian process regression and Sobol sampling.

import warnings
from decimal import Decimal, ROUND_DOWN

import numpy as np
import pandas as pd
from scipy.stats import qmc, norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

warnings.filterwarnings("ignore", module="sklearn")

In [2]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

DATA_PATH = "../data/data.csv"
FUNCTION_ID = 6

# round_trip parsing keeps float64 values exactly as written in the CSV
data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[data["function"] == FUNCTION_ID].copy()

feature_cols = [
    f"x{i}" for i in range(1, 9)
    if function_data[f"x{i}"].notna().any()
]

# Modelling arrays
X = function_data[feature_cols].to_numpy()
y = function_data["y"].to_numpy()

# Display dataframe
df = function_data[[*feature_cols, "y", "round"]].copy()
df["Stage"] = df["round"].apply(
    lambda r: "Initial Data" if r == 0 else f"Week {r}"
)
df = df.rename(columns={"round": "Round"})
df = df[[*feature_cols, "y", "Stage", "Round"]]

# Display formatting only — underlying float64 values are unchanged
pd.set_option(
    "display.float_format",
    lambda v: f"{v:.6e}" if abs(v) < 1e-3 and v != 0 else f"{v:.6f}"
)

print(f"Function {FUNCTION_ID} observations: {len(df)}")
print(f"Active features: {feature_cols}")
print(f"Dimensions: {len(feature_cols)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Rounds: R0–R{df['Round'].max()}")

display(df)

Function 6 observations: 31
Active features: ['x1', 'x2', 'x3', 'x4', 'x5']
Dimensions: 5
X shape: (31, 5)
y shape: (31,)
Rounds: R0–R11


,x1,x2,x3,x4,x5,y,Stage,Round
85,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265,Initial Data,0
86,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955,Initial Data,0
87,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200,Initial Data,0
88,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058,Initial Data,0
89,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237,Initial Data,0
90,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049,Initial Data,0
91,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786,Initial Data,0
92,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343,Initial Data,0
93,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170,Initial Data,0
94,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116,Initial Data,0


In [3]:
# =============================================================================
# TuRBO-1 — BAYESIAN OPTIMISATION
# =============================================================================
# Proposes the next Function 6 query (maximisation on [0, 1]^d).
#   1. Fit an ARD Matérn-5/2 Gaussian process to all observations.
#   2. Replay the weekly history to rebuild the trust-region box length
#      (double after SUCC_TOL successes, halve after FAIL_TOL failures).
#   3. Centre an ARD-shaped box on the best observed point, then generate
#      Sobol candidates inside it plus local perturbations of the incumbent.
#   4. Score candidates with the GP: Thompson sample, posterior mean and
#      Expected Improvement (EI). The EI maximiser is the recommended query.

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------

LENGTH_INIT = 0.08     # initial box side length (fraction of the unit cube)
LENGTH_MIN = 0.5 ** 7  # box is reset to LENGTH_INIT if it shrinks below this
LENGTH_MAX = 0.40
SUCC_TOL = 3           # consecutive improving rounds before the box doubles
FAIL_TOL = None        # None -> max(4, d) failing rounds before the box halves
SUCCESS_EPS = 1e-6     # minimum gain that counts as an improvement

N_CAND = 8192          # Sobol candidates inside the trust region
N_LOCAL = 2048         # Gaussian perturbations around the incumbent
LOCAL_SCALE = 0.015    # std of those perturbations
N_TS = 1024            # candidates entering the Thompson-sampling draw


# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------

def truncate6(value):
    """Truncate (not round) to six decimals, as required by the submission format."""
    return str(
        Decimal(repr(float(value))).quantize(Decimal("0.000001"), rounding=ROUND_DOWN)
    )


def format_point(x):
    """Format a point as the dash-separated submission string."""
    return "-".join(truncate6(v) for v in x)


# -----------------------------------------------------------------------------
# TuRBO-1
# -----------------------------------------------------------------------------

def turbo_next(X, y, rounds, x_cols, seed=42):
    """Return the next-query candidates for observations (X, y) taken in `rounds`."""
    n, d = X.shape
    rng = np.random.default_rng(seed)
    fail_tol = max(4, d) if FAIL_TOL is None else FAIL_TOL

    # 1. Fit the Gaussian process on standardised y
    y_mu, y_sd = y.mean(), y.std()
    y_scaled = (y - y_mu) / y_sd

    kernel = (
        ConstantKernel(1.0, (1e-2, 1e2))
        * Matern(length_scale=[0.3] * d, length_scale_bounds=(0.01, 10), nu=2.5)
        + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-6, 1.0))
    )
    gp = GaussianProcessRegressor(
        kernel=kernel, normalize_y=False, n_restarts_optimizer=15, random_state=seed
    )
    gp.fit(X, y_scaled)

    ls = np.atleast_1d(gp.kernel_.k1.k2.length_scale)
    noise_sd = np.sqrt(gp.kernel_.k2.noise_level) * y_sd

    # 2. Replay the weekly history to reconstruct the trust-region length
    order = np.argsort(rounds, kind="stable")
    r_sorted, y_sorted = rounds[order], y[order]

    first_round = r_sorted.min()
    best = y_sorted[r_sorted == first_round].max()
    length, succ, fail = LENGTH_INIT, 0, 0

    print("Round | Best y       | Result  | Box length")
    for r in np.unique(r_sorted[r_sorted > first_round]):
        yr = y_sorted[r_sorted == r].max()
        improved = yr > best + SUCCESS_EPS

        if improved:
            best = yr
            succ, fail = succ + 1, 0
        else:
            succ, fail = 0, fail + 1

        if succ >= SUCC_TOL:
            length = min(2 * length, LENGTH_MAX)
            succ = 0
        elif fail >= fail_tol:
            length /= 2
            fail = 0

        if length < LENGTH_MIN:
            length, succ, fail = LENGTH_INIT, 0, 0

        print(f"{r:>5} | {yr:>12.9f} | {'Success' if improved else 'Failure':<7} | {length:.4f}")

    # 3. ARD-weighted trust region centred on the best observed point
    incumbent = X[np.argmax(y)].copy()
    weights = ls / ls.mean()
    weights /= np.prod(weights) ** (1 / d)
    lb = np.clip(incumbent - weights * length / 2, 0, 1)
    ub = np.clip(incumbent + weights * length / 2, 0, 1)

    # 4. Candidates: Sobol points (each perturbing a random subset of
    #    coordinates), local perturbations of the incumbent, and the incumbent
    sobol = qmc.Sobol(d=d, scramble=True, seed=seed)
    perturbed = lb + (ub - lb) * sobol.random(N_CAND)

    mask = rng.random((N_CAND, d)) <= min(1.0, 20.0 / d)
    empty = np.where(~mask.any(axis=1))[0]
    if len(empty):
        mask[empty, rng.integers(d, size=len(empty))] = True

    sobol_candidates = np.where(mask, perturbed, incumbent)
    local = np.clip(incumbent + rng.normal(0, LOCAL_SCALE, size=(N_LOCAL, d)), lb, ub)
    candidates = np.vstack([sobol_candidates, local, incumbent[None, :]])

    # 5. GP predictions and Expected Improvement (original y scale)
    mu_scaled, sd_scaled = gp.predict(candidates, return_std=True)
    mu, sd = mu_scaled * y_sd + y_mu, sd_scaled * y_sd

    improvement = mu - y.max()
    z = np.divide(improvement, sd, out=np.zeros_like(sd), where=sd > 0)
    ei = improvement * norm.cdf(z) + sd * norm.pdf(z)
    ei = np.where(sd > 0, ei, np.maximum(improvement, 0))

    # 6. Thompson sampling over the EI/mean maximisers plus a random subset
    mean_idx, ei_idx = np.argmax(mu), np.argmax(ei)
    priority = np.unique([ei_idx, mean_idx])
    remaining = np.setdiff1d(np.arange(len(candidates)), priority)
    n_extra = min(N_TS - len(priority), len(remaining))
    selected = np.r_[priority, rng.choice(remaining, n_extra, replace=False)]

    ts_draw = gp.sample_y(candidates[selected], n_samples=1, random_state=seed).ravel()
    ts_idx = selected[np.argmax(ts_draw)]

    # 7. Report each candidate (truncated to six decimals) with its GP prediction
    def report(label, idx):
        x = np.array([float(truncate6(v)) for v in candidates[idx]])
        pred_scaled, std_scaled = gp.predict(x[None, :], return_std=True)
        print(f"\n{label}")
        print(format_point(x))
        print(f"Predicted y: {pred_scaled[0] * y_sd + y_mu:.9f} +/- {std_scaled[0] * y_sd:.9f}")
        print(f"Distance from incumbent: {np.linalg.norm(x - incumbent):.6f}")
        return x

    print(f"\nObservations: {n} | Best observed y: {y.max():.9f}")
    print(f"Incumbent: {format_point(incumbent)}")
    print(f"Kernel: {gp.kernel_}")
    print(f"GP fitted noise SD: {noise_sd:.6f}")
    print(f"Box length: {length:.4f} (success {succ}/{SUCC_TOL}, failure {fail}/{fail_tol})")
    print("\nTrust region:")
    for name, lo, hi in zip(x_cols, lb, ub):
        print(f"  {name}: [{lo:.4f}, {hi:.4f}]")

    x_ts = report("Thompson sampling", ts_idx)
    x_mean = report("Best posterior mean", mean_idx)
    x_ei = report("Best expected improvement", ei_idx)

    print(f"\nRecommended next query (Expected Improvement):\n{format_point(x_ei)}")

    return {
        "x_next": x_ei,
        "x_next_str": [truncate6(v) for v in x_ei],
        "x_ts": x_ts,
        "x_mean": x_mean,
        "x_ei": x_ei,
        "gp": gp,
        "incumbent": incumbent,
        "length": length,
        "lb": lb,
        "ub": ub,
    }


# -----------------------------------------------------------------------------
# RUN
# -----------------------------------------------------------------------------

rounds = function_data["round"].to_numpy()

result4 = turbo_next(X, y, rounds, feature_cols, seed=RANDOM_STATE)

x_next4 = result4["x_next"]
x_next4_str = result4["x_next_str"]

Round | Best y       | Result  | Box length
    1 | -0.305729456 | Success | 0.0800
    2 | -1.175428372 | Failure | 0.0800
    3 | -0.410163523 | Failure | 0.0800
    4 | -0.757140109 | Failure | 0.0800
    5 | -0.278162833 | Success | 0.0800
    6 | -0.304925466 | Failure | 0.0800
    7 | -0.237329909 | Success | 0.0800
    8 | -0.385476478 | Failure | 0.0800
    9 | -0.068174249 | Success | 0.0800
   10 | -0.071432425 | Failure | 0.0800
   11 |  0.005978097 | Success | 0.0800



Observations: 31 | Best observed y: 0.005978097
Incumbent: 0.429654-0.391293-0.666659-0.751633-0.055887
Kernel: 1.55**2 * Matern(length_scale=[0.793, 0.976, 2.16, 0.999, 1.68], nu=2.5) + WhiteKernel(noise_level=0.00583)
GP fitted noise SD: 0.051850
Box length: 0.0800 (success 1/3, failure 0/5)

Trust region:
  x1: [0.4039, 0.4555]
  x2: [0.3595, 0.4230]
  x3: [0.5964, 0.7369]
  x4: [0.7191, 0.7841]
  x5: [0.0011, 0.1107]

Thompson sampling
0.437740-0.416491-0.733923-0.772188-0.009227
Predicted y: 0.021978360 +/- 0.071642700
Distance from incumbent: 0.088456

Best posterior mean
0.444147-0.417600-0.736303-0.723905-0.012236
Predicted y: 0.050114811 +/- 0.077046175
Distance from incumbent: 0.091797

Best expected improvement
0.444147-0.417600-0.736303-0.723905-0.012236
Predicted y: 0.050114811 +/- 0.077046175
Distance from incumbent: 0.091797

Recommended next query (Expected Improvement):
0.444147-0.417600-0.736303-0.723905-0.012236
